# Stream Live Earthquake Data to Delta Lake

This notebook demonstrates streaming earthquake data from the USGS GeoJSON API into a Delta Lake table using only the deltalake package.

## Import Required Libraries

In [ ]:
from deltalake import write_deltalake, DeltaTable
import pandas as pd
from pandas import StringDtype
from pathlib import Path
import requests
import time
from datetime import datetime, timezone

## Configure Paths and API

In [2]:
ROOT = Path("/home/blaine/wsl_git/data_sci_demo")
DELTA_DIR = ROOT / "notebooks" / "data" / "earthquakes_delta_streamed"

# Create the delta directory
DELTA_DIR.mkdir(parents=True, exist_ok=True)

# USGS GeoJSON API endpoint
USGS_GEOJSON_URL = (
    "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_hour.geojson"
)

print(f"Delta directory: {DELTA_DIR}")
print(f"API endpoint: {USGS_GEOJSON_URL}")

Delta directory: /home/blaine/wsl_git/data_sci_demo/notebooks/data/earthquakes_delta_streamed
API endpoint: https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_hour.geojson


## Define Helper Functions

In [3]:
def normalize_feature(feature):
    """Extract normalized earthquake data from USGS GeoJSON feature"""
    props = feature.get("properties", {})
    geom = feature.get("geometry", {})
    coords = geom.get("coordinates", [None, None, None])
    ts = props.get("time")
    dt = datetime.fromtimestamp(ts / 1000, tz=timezone.utc) if ts else None
    return {
        "id": feature.get("id"),
        "date": dt.date() if dt else None,
        "time_utc": dt,
        "magnitude": props.get("mag"),
        "mag_type": props.get("magType"),
        "type": props.get("type"),
        "status": props.get("status"),
        "place": props.get("place"),
        "tsunami": props.get("tsunami"),
        "significance": props.get("sig"),
        "net": props.get("net"),
        "code": props.get("code"),
        "ids": props.get("ids"),
        "sources": props.get("sources"),
        "types": props.get("types"),
        "nst": props.get("nst"),
        "dmin": props.get("dmin"),
        "rms": props.get("rms"),
        "gap": props.get("gap"),
        "alert": props.get("alert"),
        "url": props.get("url"),
        "detail": props.get("detail"),
        "depth_km": coords[2] if len(coords) > 2 else None,
        "longitude": coords[0],
        "latitude": coords[1],
    }


def fetch_events(timeout=10):
    """Fetch current earthquake events from USGS feed"""
    try:
        r = requests.get(USGS_GEOJSON_URL, timeout=timeout)
        r.raise_for_status()
        data = r.json()
        return [normalize_feature(f) for f in data.get("features", [])]
    except Exception as e:
        print(f"[warn] fetch error: {e}")
        return []


print("Helper functions loaded.")

Helper functions loaded.


## Stream Configuration

In [4]:
POLL_INTERVAL = 15  # seconds between polls
MAX_ITERATIONS = 10  # number of polls (set to None for unlimited)

print("Stream configuration:")
print(f"  - Poll interval: {POLL_INTERVAL} seconds")
print(f"  - Max iterations: {MAX_ITERATIONS}")
print("  - Press interrupt kernel to stop early")

Stream configuration:
  - Poll interval: 15 seconds
  - Max iterations: 10
  - Press interrupt kernel to stop early


## Stream Live Data to Delta Lake

Stream earthquake data from the USGS API and write to Delta Lake in real-time.

In [ ]:
delta_path = str(DELTA_DIR)
events = {}  # Track unique events by ID

# Load existing earthquake IDs from Delta table if it exists
try:
    existing_dt = DeltaTable(delta_path)
    existing_df = existing_dt.to_pandas()
    existing_ids = set(existing_df["id"].dropna().tolist())
    # Populate events dictionary with existing data
    for _, row in existing_df.iterrows():
        events[row["id"]] = row.to_dict()
    print(f"Loaded {len(existing_ids)} existing earthquake IDs from Delta table")
    print(f"Current Delta table version: {existing_dt.version()}")
except Exception:
    existing_ids = set()
    print("Starting fresh - no existing Delta table found")

iteration = 0

# Helper to coerce dtypes to stable schema


def coerce_schema(df: pd.DataFrame) -> pd.DataFrame:
    # Ensure datetime with UTC
    if "time_utc" in df.columns:
        df["time_utc"] = pd.to_datetime(df["time_utc"], utc=True, errors="coerce")
    # Ensure date is ISO string (partition field)
    if "date" in df.columns:
        df["date"] = df["date"].astype("string")
    # String fields
    for col in [
        "id",
        "place",
        "mag_type",
        "type",
        "status",
        "net",
        "code",
        "ids",
        "sources",
        "types",
        "alert",
        "url",
        "detail",
    ]:
        if col in df.columns:
            df[col] = df[col].astype(StringDtype())
    # Numeric floats
    for col in [
        "magnitude",
        "depth_km",
        "longitude",
        "latitude",
        "dmin",
        "rms",
        "gap",
        "significance",
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    # Integers (nullable)
    for col in ["nst", "tsunami"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    return df


try:
    while MAX_ITERATIONS is None or iteration < MAX_ITERATIONS:
        iteration += 1

        # Fetch new events from API
        new_feats = fetch_events()
        new_count = 0

        for ev in new_feats:
            eid = ev.get("id")
            if eid and eid not in events:
                events[eid] = ev
                new_count += 1

        print(f"[Poll {iteration}] Fetched {len(new_feats)} events, {new_count} new")

        # Write new events to Delta Lake
        if new_count > 0:
            # Get the new events
            new_event_list = [events[eid] for eid in list(events.keys())[-new_count:]]
            df_new = pd.DataFrame(new_event_list)

            # Coerce dtypes to avoid Null-type schema errors
            df_new = coerce_schema(df_new)

            # Write to Delta Lake
            if len(existing_ids) == 0 and iteration == 1:
                # First write ever: create table
                write_deltalake(
                    delta_path, df_new, mode="overwrite", partition_by=["date"]
                )
                print(f"  ✓ Created Delta table with {len(df_new)} rows")
            else:
                # Append new data
                write_deltalake(
                    delta_path, df_new, mode="append", partition_by=["date"]
                )
                print(f"  ✓ Appended {len(df_new)} rows to Delta table")

        print(f"  Total unique events tracked: {len(events)}\n")

        # Wait before next poll
        if MAX_ITERATIONS is None or iteration < MAX_ITERATIONS:
            time.sleep(POLL_INTERVAL)

except KeyboardInterrupt:
    print(f"\nStream interrupted by user after {iteration} polls.")

print(f"\n✓ Streaming complete! Total unique events: {len(events)}")

Loaded 9 existing earthquake IDs from Delta table
Current Delta table version: 1
[Poll 1] Fetched 9 events, 2 new
[Poll 1] Fetched 9 events, 2 new
  ✓ Appended 2 rows to Delta table
  Total unique events tracked: 11

  ✓ Appended 2 rows to Delta table
  Total unique events tracked: 11

[Poll 2] Fetched 9 events, 0 new
  Total unique events tracked: 11

[Poll 2] Fetched 9 events, 0 new
  Total unique events tracked: 11


Stream interrupted by user after 2 polls.

✓ Streaming complete! Total unique events: 11

Stream interrupted by user after 2 polls.

✓ Streaming complete! Total unique events: 11


## Verify Delta Lake Table

Read back the Delta Lake table to verify the streamed data.

In [6]:
# Load the Delta table
dt = DeltaTable(delta_path)

# Get row count and display data
df_verify = dt.to_pandas()
print(f"Total rows in Delta table: {len(df_verify)}")
print(f"Delta table version: {dt.version()}")

# Show sample data
print("\nSample data (first 5 rows):")
df_verify.head()

Total rows in Delta table: 11
Delta table version: 2

Sample data (first 5 rows):


,id,date,time_utc,magnitude,mag_type,type,status,place,tsunami,significance,...,nst,dmin,rms,gap,alert,url,detail,depth_km,longitude,latitude
0,ci41350040,2025-12-12,2025-12-12 02:21:12.400000+00:00,0.65,ml,earthquake,automatic,"19 km SW of Ocotillo Wells, CA",0,6,...,31,0.075950,0.21,62,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,6.860000,-116.290667,33.030000
1,us6000rubp,2025-12-12,2025-12-12 01:54:12.434000+00:00,3.10,ml,earthquake,reviewed,"109 km N of Yakutat, Alaska",0,148,...,25,0.478000,1.14,78,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,5.000000,-139.864800,60.531800
2,hv74850332,2025-12-12,2025-12-12 02:16:50.140000+00:00,1.72,md,earthquake,automatic,"1 km SSW of Pāhala, Hawaii",0,46,...,27,0.043380,0.13,112,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,33.200001,-155.483505,19.194000
3,nc75278541,2025-12-12,2025-12-12 02:13:47.960000+00:00,0.72,md,earthquake,automatic,"8 km NW of The Geysers, CA",0,8,...,10,0.008801,0.02,85,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,1.730000,-122.814499,38.831501
4,nc75278536,2025-12-12,2025-12-12 01:57:14.450000+00:00,2.47,md,earthquake,automatic,"9 km SSE of San Juan Bautista, CA",0,94,...,57,0.045050,0.20,36,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,6.290000,-121.509834,36.770668


In [8]:
# Summary statistics
print("Magnitude statistics:")
print(df_verify["magnitude"].describe())

print("\nDepth statistics (km):")
print(df_verify["depth_km"].describe())

print("\nTop 5 largest earthquakes:")
top5 = df_verify.nlargest(5, "magnitude")[
    ["time_utc", "magnitude", "depth_km", "place"]
]
top5

Magnitude statistics:
count    11.00000
mean      1.50000
std       1.00449
min       0.53000
25%       0.74000
50%       1.00000
75%       2.09500
max       3.28000
Name: magnitude, dtype: float64

Depth statistics (km):
count    11.000000
mean      9.558182
std       9.513504
min       1.610000
25%       4.975000
50%       6.290000
75%      10.360000
max      33.200001
Name: depth_km, dtype: float64

Top 5 largest earthquakes:


,time_utc,magnitude,depth_km,place
10,2025-12-12 01:19:44.410000+00:00,3.28,6.460000,"9 km SSE of San Juan Bautista, CA"
1,2025-12-12 01:54:12.434000+00:00,3.10,5.000000,"109 km N of Yakutat, Alaska"
4,2025-12-12 01:57:14.450000+00:00,2.47,6.290000,"9 km SSE of San Juan Bautista, CA"
2,2025-12-12 02:16:50.140000+00:00,1.72,33.200001,"1 km SSW of Pāhala, Hawaii"
6,2025-12-12 01:47:11.270000+00:00,1.27,20.180000,"11 km N of Piru, CA"
